In [0]:
# Ingest product data into the Bronze Delta table
source_path = "abfss://source@stecomdev0011.dfs.core.windows.net/products/"
schema_path = "abfss://bronze@stecomdev0011.dfs.core.windows.net/_schemas/products/"
checkpoint_path = "abfss://bronze@stecomdev0011.dfs.core.windows.net/_checkpoints/products/"

df_products = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", schema_path)
        .load(source_path)
)

query_products = (
    df_products.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable("ecom_dev.bronze.products")
)

query_products.awaitTermination()

In [0]:
# Read from Bronze table and display
df_products_bronze = spark.table("ecom_dev.bronze.products")
display(df_products_bronze)